In [1]:
!pip install datasets sentence-transformers faiss-cpu ollama ragas numpy \
             langchain-ollama langchain-community langchain networkx

In [2]:
import json
import os
import time
import random
import traceback
from datetime import datetime

import numpy as np
import faiss
import ollama
import networkx as nx
from datasets import load_dataset, Dataset
from sentence_transformers import SentenceTransformer
from ragas import evaluate
from ragas.metrics import context_recall, context_precision, faithfulness
from langchain_ollama import ChatOllama
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.embeddings import HuggingFaceEmbeddings

print("All imports OK")
print(f"NetworkX version: {nx.__version__}")

All imports OK
NetworkX version: 3.4.2


/var/folders/wn/m7ypjd8j3y5fc5flk1tx2hl40000gn/T/ipykernel_97404/3788186831.py:15: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_recall, context_precision, faithfulness
/var/folders/wn/m7ypjd8j3y5fc5flk1tx2hl40000gn/T/ipykernel_97404/3788186831.py:15: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_recall, context_precision, faithfulness
/var/folders/wn/m7ypjd8j3y5fc5flk1tx2hl40000gn/T/ipykernel_97404/3788186831.py:15: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. 

In [3]:
# ── CONFIG
# FIXED across V0, V1, V2
EMBED_MODEL    = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL      = "llama3.2:3b"
TEMPERATURE    = 0

# CHANGED from V1 — wider initial retrieval for graph filter
TOP_K_INITIAL  = 10
TOP_K_FINAL    = 5

# Graph constraint
HOP_LIMIT       = 2
PROXIMITY_BONUS = 0.3
PENALTY         = 0.5

# Sampling
N_PILOT        = 10      # ← added
RANDOM_SEED    = 42      # ← added

# Paths
DATA_DIR        = "data"
os.makedirs(DATA_DIR, exist_ok=True)
QUERIES_FILE    = os.path.join(DATA_DIR, "queries.json")
V1_CHUNKS_FILE  = os.path.join(DATA_DIR, "v1_chunks.json")
V1_INDEX_FILE   = os.path.join(DATA_DIR, "v1_index.faiss")
GRAPH_FILE      = os.path.join(DATA_DIR, "v2_graph.pkl")
RESULTS_FILE    = os.path.join(DATA_DIR, "v2_results.json")

# Prompt — FIXED
SYSTEM_PROMPT = (
    "You are a research assistant. Answer the question using ONLY the "
    "provided context. If the context does not contain enough information, "
    "say: I cannot answer this question from the provided context. "
    "Be concise and precise."
)

print("Config set.")
print(f"  Initial retrieval K: {TOP_K_INITIAL}")
print(f"  Final context K:     {TOP_K_FINAL}")
print(f"  Hop limit:           {HOP_LIMIT}")
print(f"  N_PILOT:             {N_PILOT}")

Config set.
  Initial retrieval K: 10
  Final context K:     5
  Hop limit:           2
  N_PILOT:             10


In [4]:
# Load embedding model
print(f"Loading embedding model: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)
print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")

# Load V1 chunks
print(f"\nLoading V1 chunks from {V1_CHUNKS_FILE}...")
with open(V1_CHUNKS_FILE) as f:
    all_chunks = json.load(f)
print(f"Chunks loaded: {len(all_chunks):,}")

# Load V1 FAISS index
print(f"\nLoading V1 FAISS index from {V1_INDEX_FILE}...")
index = faiss.read_index(V1_INDEX_FILE)
print(f"Index vectors: {index.ntotal:,}")

# Load queries — same file as V0 and V1
print(f"\nLoading queries from {QUERIES_FILE}...")
with open(QUERIES_FILE) as f:
    queries = json.load(f)
print(f"Queries loaded: {len(queries)}")

# RAGAS — local Ollama judge
from ragas import RunConfig
ragas_llm = LangchainLLMWrapper(
    ChatOllama(model=LLM_MODEL, temperature=TEMPERATURE)
)
ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name=EMBED_MODEL)
)
run_config = RunConfig(timeout=180, max_retries=3, max_wait=60)
print("\nRAGAS configured.")

print("\nAll artefacts loaded.")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 384

Loading V1 chunks from data/v1_chunks.json...
Chunks loaded: 46,882

Loading V1 FAISS index from data/v1_index.faiss...
Index vectors: 46,882

Loading queries from data/queries.json...
Queries loaded: 10


/var/folders/wn/m7ypjd8j3y5fc5flk1tx2hl40000gn/T/ipykernel_97404/745069602.py:25: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(
/var/folders/wn/m7ypjd8j3y5fc5flk1tx2hl40000gn/T/ipykernel_97404/745069602.py:29: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  HuggingFaceEmbeddings(model_name=EMBED_MODEL)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



RAGAS configured.

All artefacts loaded.


/var/folders/wn/m7ypjd8j3y5fc5flk1tx2hl40000gn/T/ipykernel_97404/745069602.py:28: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(


In [5]:
def build_document_graph(chunks: list) -> nx.Graph:
    """
    Build a NetworkX graph over the QASPER corpus.

    Nodes: each chunk is one node, keyed by chunk_id.
    Node attributes: paper_id, section_name, paragraph_index, chunk_text.

    Edge Type 1 — Sequential Adjacency:
        Connects paragraph N to paragraph N+1 within the same section
        of the same paper. Encodes narrative continuity.
        attribute: type='sequential'

    Edge Type 2 — Section Membership:
        Connects all paragraphs within the same section of the same paper
        to a virtual section node (not a real paragraph).
        attribute: type='section'

    No inter-paper edges — queries target individual papers in QASPER.
    """
    G = nx.Graph()

    # Add all paragraph nodes
    for c in chunks:
        G.add_node(
            c["chunk_id"],
            paper_id        = c["paper_id"],
            section_name    = c["section_name"],
            paragraph_index = c["paragraph_index"],
            chunk_text      = c["chunk_text"],
        )

    # Group chunks by (paper_id, section_name) for edge construction
    from collections import defaultdict
    sections = defaultdict(list)
    for c in chunks:
        key = (c["paper_id"], c["section_name"])
        sections[key].append(c)

    # Sort each section by paragraph_index
    for key in sections:
        sections[key].sort(key=lambda x: x["paragraph_index"])

    virtual_node_id = len(chunks)  # virtual section nodes start after all real nodes

    for (paper_id, section_name), section_chunks in sections.items():

        # Edge Type 1 — Sequential adjacency within section
        for i in range(len(section_chunks) - 1):
            G.add_edge(
                section_chunks[i]["chunk_id"],
                section_chunks[i+1]["chunk_id"],
                type="sequential",
            )

        # Edge Type 2 — Section membership via virtual node
        virtual_id = virtual_node_id
        virtual_node_id += 1
        G.add_node(virtual_id, type="section_virtual",
                   paper_id=paper_id, section_name=section_name)
        for c in section_chunks:
            G.add_edge(c["chunk_id"], virtual_id, type="section")

    return G


print("Building document graph...")
t0 = time.time()
G  = build_document_graph(all_chunks)
elapsed = time.time() - t0

print(f"Done in {elapsed:.1f}s")
print(f"  Nodes: {G.number_of_nodes():,}")
print(f"  Edges: {G.number_of_edges():,}")
print(f"  Real paragraph nodes: {len(all_chunks):,}")
print(f"  Virtual section nodes: {G.number_of_nodes() - len(all_chunks):,}")

# Save graph
import pickle
with open(GRAPH_FILE, "wb") as f:
    pickle.dump(G, f)
size_mb = os.path.getsize(GRAPH_FILE) / (1024*1024)
print(f"\nGraph saved → {GRAPH_FILE} ({size_mb:.1f} MB)")

Building document graph...
Done in 0.2s
  Nodes: 59,300
  Edges: 81,346
  Real paragraph nodes: 46,882
  Virtual section nodes: 12,418

Graph saved → data/v2_graph.pkl (26.0 MB)


In [6]:
# Pick first paper and inspect its subgraph
test_paper_id = all_chunks[0]["paper_id"]
test_chunks   = [c for c in all_chunks if c["paper_id"] == test_paper_id]

print(f"Paper: {test_paper_id}")
print(f"Chunks in paper: {len(test_chunks)}")
print()

# Check sequential edges on first two chunks
c0 = test_chunks[0]
c1 = test_chunks[1]
print(f"Chunk 0: {c0['section_name']} | para {c0['paragraph_index']}")
print(f"Chunk 1: {c1['section_name']} | para {c1['paragraph_index']}")
print(f"Edge between them: {G.has_edge(c0['chunk_id'], c1['chunk_id'])}")
print(f"Edge type: {G.get_edge_data(c0['chunk_id'], c1['chunk_id'])}")
print()

# Check hop distance
c2 = test_chunks[2]
print(f"Chunk 2: {c2['section_name']} | para {c2['paragraph_index']}")
try:
    dist_0_1 = nx.shortest_path_length(G, c0["chunk_id"], c1["chunk_id"])
    dist_0_2 = nx.shortest_path_length(G, c0["chunk_id"], c2["chunk_id"])
    print(f"Hop distance chunk 0 → chunk 1: {dist_0_1}")
    print(f"Hop distance chunk 0 → chunk 2: {dist_0_2}")
except nx.NetworkXNoPath:
    print("No path found")
print()

# Check a chunk from a different paper
other_chunk = next(c for c in all_chunks if c["paper_id"] != test_paper_id)
print(f"Different paper chunk: {other_chunk['paper_id']} | chunk_id {other_chunk['chunk_id']}")
try:
    dist = nx.shortest_path_length(G, c0["chunk_id"], other_chunk["chunk_id"])
    print(f"Hop distance to different paper: {dist}")
except nx.NetworkXNoPath:
    print(f"No path to different paper chunk — correct, no inter-paper edges")

Paper: 1909.00694
Chunks in paper: 56

Chunk 0: Introduction | para 0
Chunk 1: Introduction | para 1
Edge between them: True
Edge type: {'type': 'sequential'}

Chunk 2: Introduction | para 2
Hop distance chunk 0 → chunk 1: 1
Hop distance chunk 0 → chunk 2: 2

Different paper chunk: 2003.07723 | chunk_id 56
No path to different paper chunk — correct, no inter-paper edges


# Graph-constrained retrieval function.

In [8]:
def retrieve_v2(
    query:      str,
    model,
    index,
    chunks:     list,
    G:          nx.Graph,
    k_initial:  int = TOP_K_INITIAL,
    k_final:    int = TOP_K_FINAL,
    hop_limit:  int = HOP_LIMIT,
) -> tuple:
    """
    V2 graph-constrained retrieval.

    Step 1 — Widened vector search: retrieve k_initial=10 candidates.
    Step 2 — Anchor selection: top-1 result is the anchor node.
    Step 3 — Graph re-scoring: score remaining candidates by hop
              distance to anchor. Within hop_limit → bonus.
              No path within hop_limit → penalty.
    Step 4 — Adjacency expansion: pull in sequential neighbours of
              anchor not already in candidate set.
    Step 5 — Select final top-k_final from re-scored candidates.

    Returns:
        retrieved_texts  — list of chunk texts for LLM context
        provenance       — list of dicts with chunk metadata + graph distance
    """

    # Step 1 — Widened vector search
    q_vec = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    distances, indices = index.search(q_vec, k_initial)

    # FAISS returns L2 distances on normalised vectors
    # Convert to cosine similarity: sim = 1 - (L2^2 / 2)
    candidates = []
    for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        if idx == -1:
            continue
        cosine_sim = float(1 - dist / 2)
        candidates.append({
            "chunk_id":   chunks[idx]["chunk_id"],
            "chunk_idx":  idx,
            "chunk_text": chunks[idx]["chunk_text"],
            "paper_id":   chunks[idx]["paper_id"],
            "section_name": chunks[idx]["section_name"],
            "paragraph_index": chunks[idx]["paragraph_index"],
            "cosine_sim": cosine_sim,
            "rank":       rank,
        })

    if not candidates:
        return [], []

    # Step 2 — Anchor is top-1
    anchor    = candidates[0]
    anchor_id = anchor["chunk_id"]

    # Step 3 — Graph re-scoring
    rescored = []
    for c in candidates:
        score = c["cosine_sim"]
        graph_dist = None

        if c["chunk_id"] == anchor_id:
            # Anchor keeps its score
            graph_dist = 0
        elif G.has_node(c["chunk_id"]) and G.has_node(anchor_id):
            try:
                path_len = nx.shortest_path_length(
                    G, anchor_id, c["chunk_id"]
                )
                graph_dist = path_len
                if path_len <= hop_limit:
                    score = score + PROXIMITY_BONUS
                else:
                    score = score * PENALTY
            except nx.NetworkXNoPath:
                # No structural connection — penalise
                graph_dist = None
                score = score * PENALTY
        else:
            score = score * PENALTY

        rescored.append({**c, "final_score": score, "graph_dist": graph_dist})

    # Step 4 — Adjacency expansion
    # Pull in sequential neighbours of anchor not already in candidate set
    existing_ids = {c["chunk_id"] for c in rescored}

    if G.has_node(anchor_id):
        for neighbour_id in G.neighbors(anchor_id):
            if neighbour_id in existing_ids:
                continue
            edge_data = G.get_edge_data(anchor_id, neighbour_id)
            if edge_data and edge_data.get("type") == "sequential":
                # Find the chunk
                neighbour_chunks = [
                    c for c in chunks if c["chunk_id"] == neighbour_id
                ]
                if neighbour_chunks:
                    nc = neighbour_chunks[0]
                    # Embed and score the neighbour
                    nc_vec = model.encode(
                        [nc["chunk_text"]],
                        convert_to_numpy=True,
                        normalize_embeddings=True,
                    ).astype("float32")
                    nc_dist = float(np.linalg.norm(q_vec - nc_vec))
                    nc_sim  = float(1 - nc_dist**2 / 2)
                    rescored.append({
                        "chunk_id":        nc["chunk_id"],
                        "chunk_idx":       nc["chunk_id"],
                        "chunk_text":      nc["chunk_text"],
                        "paper_id":        nc["paper_id"],
                        "section_name":    nc["section_name"],
                        "paragraph_index": nc["paragraph_index"],
                        "cosine_sim":      nc_sim,
                        "rank":            999,
                        "final_score":     nc_sim + PROXIMITY_BONUS,
                        "graph_dist":      1,
                    })

    # Step 5 — Select final top-k_final by final_score
    rescored.sort(key=lambda x: x["final_score"], reverse=True)
    final = rescored[:k_final]

    retrieved_texts = [c["chunk_text"] for c in final]
    provenance      = [{
        "chunk_id":        c["chunk_id"],
        "paper_id":        c["paper_id"],
        "section_name":    c["section_name"],
        "paragraph_index": c["paragraph_index"],
        "cosine_sim":      round(c["cosine_sim"], 4),
        "graph_dist":      c["graph_dist"],
        "final_score":     round(c["final_score"], 4),
    } for c in final]

    return retrieved_texts, provenance


print("retrieve_v2 defined.")

retrieve_v2 defined.


In [9]:
test_q = queries[0]["question"]
print(f"Question: {test_q}")
print()

t0 = time.time()
retrieved_texts, provenance = retrieve_v2(
    test_q, embed_model, index, all_chunks, G
)
t_retr = time.time() - t0

print(f"Retrieved {len(retrieved_texts)} chunks in {t_retr*1000:.1f}ms")
print()
print("Provenance — graph structure of retrieved chunks:")
for i, p in enumerate(provenance):
    print(f"  [{i+1}] {p['paper_id']} | {p['section_name']} | para {p['paragraph_index']}")
    print(f"       cosine={p['cosine_sim']}  graph_dist={p['graph_dist']}  final_score={p['final_score']}")

Question: Why is this work different from text-only UNMT?

Retrieved 5 chunks in 519.7ms

Provenance — graph structure of retrieved chunks:
  [1] 1904.05584 | Combining Character and Word-level Representations | para 8
       cosine=0.2576  graph_dist=1  final_score=0.5576
  [2] 1904.05584 | Combining Character and Word-level Representations | para 9
       cosine=0.4884  graph_dist=0  final_score=0.4884
  [3] 1610.04377 | Pre-Processing Modules | para 5
       cosine=0.488  graph_dist=None  final_score=0.244
  [4] 1911.03648 | Pre-Processing | para 4
       cosine=0.4606  graph_dist=None  final_score=0.2303
  [5] 1811.11365 | Implementation Details and Baseline Models | para 4
       cosine=0.4574  graph_dist=None  final_score=0.2287


In [10]:
def generate(query: str, context_chunks: list) -> str:
    context  = "\n\n---\n\n".join(context_chunks)
    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"},
        ],
        options={"temperature": TEMPERATURE},
    )
    return response["message"]["content"].strip()


def compute_provenance_coverage(provenance: list) -> float:
    """
    Provenance Coverage — V2 only metric.

    Measures what proportion of the final retrieved chunks have a
    traceable structural path to the anchor node via the graph.

    A chunk has provenance if graph_dist is not None
    (i.e. it is connected to the anchor within hop_limit hops,
    or it IS the anchor at distance 0).

    Score = chunks with graph path / total chunks
    """
    if not provenance:
        return 0.0
    with_path = sum(1 for p in provenance if p["graph_dist"] is not None)
    return round(with_path / len(provenance), 4)


def evaluate_query(question, answer, ground_truth, retrieved):
    ragas_data = Dataset.from_dict({
        "question":      [question],
        "answer":        [answer],
        "contexts":      [retrieved],
        "ground_truth":  [ground_truth],
        "ground_truths": [[ground_truth]],
    })
    result = evaluate(
        ragas_data,
        metrics=[context_recall, context_precision, faithfulness],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        raise_exceptions=False,
        run_config=run_config,
    )

    def safe_score(key):
        val   = result[key]
        score = val[0] if isinstance(val, list) else val
        return 0.0 if (score is None or
            (isinstance(score, float) and np.isnan(score))) else float(score)

    return {
        "context_recall":    safe_score("context_recall"),
        "context_precision": safe_score("context_precision"),
        "faithfulness":      safe_score("faithfulness"),
    }


print("Functions defined.")
print("Provenance Coverage metric ready.")

Functions defined.
Provenance Coverage metric ready.


In [11]:
print(f"Running V2 pilot — {N_PILOT} queries")
print("-" * 65)

results     = []
t_run_start = time.time()

for i, q in enumerate(queries):
    t_q = time.time()
    try:
        # Retrieval — V2 graph-constrained
        t0 = time.time()
        retrieved_texts, provenance = retrieve_v2(
            q["question"], embed_model, index, all_chunks, G
        )
        t_retr = time.time() - t0

        # Provenance Coverage — computed from graph metadata
        prov_coverage = compute_provenance_coverage(provenance)

        # Generation
        t0        = time.time()
        generated = generate(q["question"], retrieved_texts)
        t_gen     = time.time() - t0

        # RAGAS evaluation
        t0      = time.time()
        metrics = evaluate_query(
            q["question"], generated, q["answer"], retrieved_texts
        )
        t_eval  = time.time() - t0

        t_total = time.time() - t_q

        results.append({
            "query_index":        i,
            "paper_id":           q["paper_id"],
            "question":           q["question"],
            "ground_truth":       q["answer"],
            "answer_type":        q["answer_type"],
            "generated_answer":   generated,
            "retrieved_chunks":   retrieved_texts,
            "provenance":         provenance,
            "context_recall":     metrics["context_recall"],
            "context_precision":  metrics["context_precision"],
            "faithfulness":       metrics["faithfulness"],
            "provenance_coverage": prov_coverage,
            "timing": {
                "retrieval_s":  round(t_retr, 3),
                "generation_s": round(t_gen, 3),
                "evaluation_s": round(t_eval, 3),
                "total_s":      round(t_total, 3),
            },
            "status": "success",
        })

        print(
            f"[{i+1:2}/{N_PILOT}] {q['answer_type']:12} "
            f"recall={metrics['context_recall']:.3f}  "
            f"precision={metrics['context_precision']:.3f}  "
            f"faith={metrics['faithfulness']:.3f}  "
            f"prov={prov_coverage:.2f}  "
            f"({t_total:.0f}s)"
        )

    except Exception as e:
        print(f"[{i+1:2}/{N_PILOT}] ERROR: {e}")
        traceback.print_exc()
        results.append({
            "query_index": i,
            "status":      "error",
            "error":       str(e),
            **q
        })

t_run_total = time.time() - t_run_start
print("-" * 65)
print(f"Total: {t_run_total:.0f}s  ({t_run_total/N_PILOT:.0f}s per query)")

Running V2 pilot — 10 queries
-----------------------------------------------------------------


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Exception raised in Job[2]: OutputParserException(Invalid json output: Given a question and an answer, analyze the complexity of each sentence in the answer. Break down each sentence into one or more fully understandable statements. Ensure that no pronouns are used in any statement. Format the outputs in JSON.
Please return the output in a JSON format that complies with the following schema as specified in JSON Schema:
{\
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )
Exception raised in Job[1]: OutputParserException(Invalid json output: The provided context was indeed useful in arriving at the given answer. The context includes key information about the vector gate mechanism, which are reflected in the answer.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )


[ 1/10] extractive   recall=0.000  precision=0.000  faith=0.000  prov=0.40  (184s)


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

...

Exception raised in Job[1]: TimeoutError()


[ 9/10] abstractive  recall=0.750  precision=0.000  faith=1.000  prov=0.80  (197s)


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Exception raised in Job[1]: TimeoutError()
Exception raised in Job[2]: TimeoutError()


[10/10] extractive   recall=0.778  precision=0.000  faith=0.000  prov=0.60  (202s)
-----------------------------------------------------------------
Total: 1960s  (196s per query)


In [25]:
successful = [r for r in results if r.get("status") == "success"]

def mean(key, rows):
    vals = [r[key] for r in rows if key in r]
    return round(sum(vals)/len(vals), 4) if vals else None

def mean_timing(key, rows):
    vals = [r["timing"][key] for r in rows if "timing" in r]
    return round(sum(vals)/len(vals), 1) if vals else None

print("=" * 60)
print("V2 GRAPH-CONSTRAINED RETRIEVAL — PILOT RESULTS")
print("=" * 60)
print(f"  Queries: {len(successful)}/{len(results)} successful")
print()
print(f"  Context Recall:      {mean('context_recall', successful)}")
print(f"  Context Precision:   {mean('context_precision', successful)}  ← unreliable (3B timeouts)")
print(f"  Faithfulness:        {mean('faithfulness', successful)}  ← unreliable (3B timeouts)")
print(f"  Provenance Coverage: {mean('provenance_coverage', successful)}")
print()
print("  Comparison:")
print("    Metric              V0      V1      V2")
print(f"    Context Recall    0.475   0.436   {mean('context_recall', successful)}")
print(f"    Prov Coverage     N/A     N/A     {mean('provenance_coverage', successful)}")
print()
print(f"  Mean time/query: {mean_timing('total_s', successful)}s")
print("=" * 60)

output = {
    "summary": {
        "variant":                   "V2 — Graph-Constrained Retrieval",
        "total_queries":             len(results),
        "successful":                len(successful),
        "mean_context_recall":       mean("context_recall", successful),
        "mean_context_precision":    mean("context_precision", successful),
        "mean_faithfulness":         mean("faithfulness", successful),
        "mean_provenance_coverage":  mean("provenance_coverage", successful),
        "mean_total_s":              mean_timing("total_s", successful),
        "note": "Precision and Faithfulness unreliable — 3B model JSON failures. HPC evaluation pending."
    },
    "config": {
        "variant":        "V2",
        "embed_model":    EMBED_MODEL,
        "llm_model":      LLM_MODEL,
        "top_k_initial":  TOP_K_INITIAL,
        "top_k_final":    TOP_K_FINAL,
        "hop_limit":      HOP_LIMIT,
        "proximity_bonus": PROXIMITY_BONUS,
        "penalty":        PENALTY,
        "temperature":    TEMPERATURE,
        "n_queries":      len(queries),
    },
    "results": results,
}

with open(RESULTS_FILE, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved → {RESULTS_FILE}")

V2 GRAPH-CONSTRAINED RETRIEVAL — PILOT RESULTS
  Queries: 10/10 successful

  Context Recall:      0.498
  Context Precision:   0.0  ← unreliable (3B timeouts)
  Faithfulness:        0.2  ← unreliable (3B timeouts)
  Provenance Coverage: 0.48

  Comparison:
    Metric              V0      V1      V2
    Context Recall    0.475   0.436   0.498
    Prov Coverage     N/A     N/A     0.48

  Mean time/query: 196.0s
Saved → data/v2_results.json
